# YuE2 Studio 5.0.0 — Cloudflare Runtime

**This launches the real YuE2 Studio web interface and its Rust backend. It is not a ComfyUI workflow.**

## Kaggle settings

- Accelerator: **GPU T4 ×2** (or another supported NVIDIA GPU)
- Internet: **ON**
- Add Kaggle secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` if they are not already available.\n- No builder output attachment is needed: this notebook downloads the private runtime dataset automatically.

## Start

Run the single code cell. It will:
1. Locate and verify the builder bundle.
2. Install the small Linux runtime dependencies and Node 22.
3. Clone the upstream `v5.0.0` Studio source and install its frontend dependencies.
4. Start the real Rust music service, the YuE2 CUDA/Vulkan engine and the upstream Vite interface.
5. Enable the Studio's own authenticated remote-access mode, configure the local Vite proxy to forward its per-session bearer key, then open a Cloudflare Quick Tunnel and print the public Studio link.

Open the printed `trycloudflare.com` address. Keep the Kaggle cell running.

## First use

Use the Studio's own setup / Models screens to download the model sets and optional engines you want. The builder does not pre-download multi-gigabyte model weights. Your songs and library data are stored under `/kaggle/working/yue2-studio-data` for this Kaggle session; export or save them before the session ends.

## Important

This is a headless browser-hosted run of the upstream application, not a native Windows desktop build. The upstream UI and Rust service are used; the Vite development server proxies the Studio API to its Rust backend. Optional engines and tools may be platform- or model-licence-dependent. The Studio application is MIT; that does not make every model weight or engine commercially licensed.

Logs: `music-server.log`, `studio-ui.log`, `cloudflared.log` in `/kaggle/working`.

In [ ]:
import os, sys, json, shutil, subprocess, time, re, urllib.request, base64, io, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
DATA = WORK / "yue2-studio-data"
SHARED = WORK / "timoncool-shared"
REPO = WORK / "YuE2-Studio"
DATA.mkdir(parents=True, exist_ok=True)
SHARED.mkdir(parents=True, exist_ok=True)

# --- Automatic Kaggle Dataset retrieval (same JJP dataset-builder pattern) --
DATASET_OWNER = ""  # Leave blank to resolve from Kaggle API credentials.
DATASET_SLUG = "yue2-studio-linux-runtime"
RUNTIME = WORK / "runtime"
RUNTIME.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode("UEsDBBQAAAAIAIiVe12CrXMn7QQAAGkMAAARAAAAc3R1ZGlvX2RhdGFzZXQucHmVVttu20YQfddXDLYvVKDQlzYo4EAFFFc2giRuYDsoAsMgKHIkbU1x2d2lbFfQv3dmL6IkX5rqReTs7Nm5nDlLuWiUtqDMAP4yqh6AmbdWVvTfThqtCjS0YnHRTGWFvalWC2hyO6/kBKTf+pVee70Sp5C3do61lUVusczyRib9kx7QTwgx2lqDe2nn8CmfzSqEWlmcKHUHBguN1oDSgA/SWFnPog8tlLw5r0xKWA5TxsDdm9WP/ij+uSjv3NYsogb3bwb1lTedVpIwN5ui4/CpT9LvoDm6epnV+QJB1pCIT6Pz88/j7NvV+PJi9GUsBhBNn8bfRb+LygU95Xwp6JRApFZ1OkObRMA95yd5bf86iJu4/ZZiD1kwbEi9Q3+Cgw8FNhbG7k+q+vmTmtz4Gr/svnF5Pr8nNepzk19x3KuczqVBuGyJAQsca610shOpCDQZff24TRXINcJCGkNMSmFUlrAXB+T1xkZHgthFzc1LHKWJIDKDRt3WabfLV3iLfSnNQBqISI8ZPlisSywjGz38qJFuI3lQBze2wDqG2B6sYKYwWl3zaq93/vEDbTw6PP4F3ryBn/0wFnOlDPG/0Lkt5onGv1tJtckmjxYpgYJyl2Xunqt8glWod2cnyBue7aTpO9o3zPdu+bYX54H1YHdtmxl2ni7uSqmThtpRWzO81i0O/Ihn6s69dtRsTT5D5rFTobSU5i5ztoSROj8imjOnU40Ivw1hN79dJodauTgPfLa9F3g1FRcKQs1gqaqWxnxORFjt4h9QyU/So+kazuUH4BhS0Q9110glyKgOuSFKU8vySs0GoFrbtDajSgyoWPyolbLciGKO3qzua6Q/6mmnm6cOjvkWyRiQQdUwldpYIBK+h6adVNLMIYca72GJ2tCEsk9F2zX7dNLZhUKVdi3uLP09jx9t3gJtnrkKD7fxD0CEeN+yBz+nfM/4QEzVzsg/1OgmumZsF55gVtoKn/NxC8GpJcV2ijz0NWR5+SEN4uFiqSxUPZWzbJlXLSYiwon+tqhF66u6JE5VW5XOX6NR1XLTtrj9PXDzfh9dj67G19kff16ML1PR1ZBSWAmf24lPni4UWdLLVKwixvpgxSVa82VTyQJrg4Y8blbChX0C4vT08O1ReijWt4NdUSuR2C2dfpNfqKqv0PYSAX9vx8dwZdtSKvgs6/aBScSZwqSty4oZv95tfXqvJVHfkswl3OS0bBeNSXidOc+6PDzu+1x/glPVPIKxNMTlBjnXVk7zgu/r2irH+kj3qapK1OlGdeikhVedQAy6g0iaxW3XH6NaXWDkOG+4Ed4mbnekhNvlF1JpMv7SSfbuYt/qM1q5UPZMUQGiXlwGYehy4DjivXMCKw+8Ft2JVGi7PyY+Oq5i4HR0TP3o/R8RDeJZUIGPE3/8wGHtJM0n+sbjAxatzSfEub20XQDFfKHK5FD9+u6dB4gjKEvK4jlaPv0k40GL25YS75MOowvKpcP3DqfzH58cG98zuukxzqm3Pn9sEOYgjYmxelv2BpQIfaOqBX+7wjb3Y181ks6bOY0GqQWhkASprDDLoYuAC1wh4RNN4xnGL3UJErNZlYcggkfWNnxhlr5iSM6vxk7S/jRup/xFjOKF2KTOqIk4FP/IRjwXkD8hBBIuzJXoukRi0b2QOvidZPUPg43Oc1Rk3gtz3fsXUEsBAhQDFAAAAAgAiJV7XYKtcyftBAAAaQwAABEAAAAAAAAAAAAAAIABAAAAAHN0dWRpb19kYXRhc2V0LnB5UEsFBgAAAAABAAEAPwAAABwFAAAAAA=="))) as archive:
    for name in archive.namelist():
        assert Path(name).name == name, f"Unsafe helper path: {name}"
        (RUNTIME / name).write_bytes(archive.read(name))
sys.path.insert(0, str(RUNTIME))
import importlib
importlib.invalidate_caches()
sys.modules.pop("studio_dataset", None)
from studio_dataset import authenticated_api

api = authenticated_api()  # Read Kaggle secrets before attempting a dataset download.
dataset_owner = DATASET_OWNER or os.environ.get("KAGGLE_USERNAME") or api.get_config_value("username")
if not dataset_owner:
    raise RuntimeError("Could not resolve Kaggle username; set DATASET_OWNER.")
dataset_id = f"{dataset_owner}/{DATASET_SLUG}"
bundle_cache = WORK / "studio-runtime-dataset"
bundle_cache.mkdir(parents=True, exist_ok=True)
print(f"Downloading private runtime dataset: {dataset_id}")
api.dataset_download_files(dataset_id, path=str(bundle_cache), unzip=True, quiet=False)

# Find the manifest in the automatically downloaded dataset.
manifests = list(bundle_cache.rglob("manifest.json"))
bundle = None
for p in manifests:
    try:
        info = json.loads(p.read_text())
        if info.get("studio_version") == "5.0.0" and info.get("tag") == "v5.0.0":
            bundle = p.parent
            break
    except Exception:
        pass
if bundle is None:
    raise FileNotFoundError(f"{dataset_id} does not contain a valid YuE2 Studio v5.0.0 manifest.")

required = [bundle / "music-server", bundle / "cloudflared", bundle / "yue2-cpp/yue-server"]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Runtime dataset is incomplete: " + ", ".join(missing))

# Install runtime libraries only; the Studio service and CUDA engine were built in notebook 1.
subprocess.run(["bash", "-lc", "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2 libvulkan1 libgtk-3-0 libayatana-appindicator3-1 ffmpeg"], check=True)

# Node 22 is used for the upstream Vite UI. The Rust service and YuE2 CUDA engine come from the bundle.
node_dir = WORK / "node-v22"
if not (node_dir / "bin/node").exists():
    node_archive = WORK / "node-v22-linux-x64.tar.xz"
    urllib.request.urlretrieve("https://nodejs.org/dist/latest-v22.x/node-v22.16.0-linux-x64.tar.xz", node_archive)
    node_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xJf", str(node_archive), "-C", str(node_dir), "--strip-components=1"], check=True)
env = os.environ.copy()
env["PATH"] = f"{node_dir}/bin:/root/.cargo/bin:/usr/local/bin:/usr/bin:/bin"
env["YUE_STUDIO_PORT"] = "8791"
env["YUE_STUDIO_DATA_ROOT"] = str(DATA)
env["TIMONCOOL_SHARED_DIR"] = str(SHARED)
env["YUE_ENGINE_BIN"] = str(bundle / "yue2-cpp/yue-server")
env["YUE_ENGINE_ROOT"] = str(bundle / "yue2-cpp")
env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["DO_NOT_TRACK"] = "1"
env["STUDIO_TELEMETRY"] = "0"
env["PYTHONUNBUFFERED"] = "1"

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "v5.0.0", "https://github.com/timoncool/YuE2-Studio.git", str(REPO)], check=True)
subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], cwd=REPO / "app", env=env, check=True)

server_log = open(WORK / "music-server.log", "w")
server = subprocess.Popen([str(bundle / "music-server")], cwd=REPO, env=env, stdout=server_log, stderr=subprocess.STDOUT)
print("Starting YuE2 Studio Rust service...")
for _ in range(90):
    if server.poll() is not None:
        raise RuntimeError("Studio backend exited. Check /kaggle/working/music-server.log")
    try:
        import urllib.request
        urllib.request.urlopen("http://127.0.0.1:8791/health", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError("Studio backend did not become healthy. Check music-server.log")

# Enable the Studio's own authenticated remote-access mode. Cloudflare is a
# reverse proxy, so requests are authenticated at the Vite proxy using this
# generated per-session key; the key is never placed in the public URL.
def studio_request(method, path, payload=None):
    body = None if payload is None else json.dumps(payload).encode()
    req = urllib.request.Request("http://127.0.0.1:8791" + path, data=body, method=method,
                                 headers={"Content-Type":"application/json"} if body else {})
    with urllib.request.urlopen(req, timeout=15) as response:
        return json.loads(response.read().decode())

access = studio_request("PUT", "/v1/network", {"enabled": True})
access = studio_request("GET", "/v1/network")
access_key = access.get("key")
if not access_key:
    raise RuntimeError("Studio did not return its local network-access key; refusing to expose an unauthenticated tunnel.")

# Restart so the service binds its authenticated network listener, as required
# by the upstream network-access setting.
server.terminate()
try:
    server.wait(timeout=20)
except subprocess.TimeoutExpired:
    server.kill()
    server.wait(timeout=10)
server = subprocess.Popen([str(bundle / "music-server")], cwd=REPO, env=env, stdout=server_log, stderr=subprocess.STDOUT)
for _ in range(45):
    if server.poll() is not None:
        raise RuntimeError("Studio backend exited while enabling authenticated access. Check music-server.log")
    try:
        urllib.request.urlopen("http://127.0.0.1:8791/health", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError("Studio backend did not restart after enabling authenticated access.")

# The browser talks to Vite on the public Cloudflare hostname. Vite forwards
# API calls to the Rust service, adding the Studio's generated bearer key and
# the original host so upstream's reverse-proxy origin checks remain enabled.
vite_config = REPO / "app/vite.config.ts"
vite_text = vite_config.read_text()
proxy_hook = """changeOrigin: true,
          configure: (proxy) => {
            proxy.on('proxyReq', (proxyReq, req) => {
              proxyReq.setHeader('Authorization', 'Bearer """ + access_key + """');
              proxyReq.setHeader('x-forwarded-host', req.headers.host || 'localhost');
            });
          },"""
if "proxy.on('proxyReq'" not in vite_text:
    vite_text = vite_text.replace("changeOrigin: true,", proxy_hook)
    vite_config.write_text(vite_text)

ui_log = open(WORK / "studio-ui.log", "w")
ui = subprocess.Popen(["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "3791", "--strictPort"],
                      cwd=REPO / "app", env=env, stdout=ui_log, stderr=subprocess.STDOUT)
for _ in range(45):
    if ui.poll() is not None:
        raise RuntimeError("Studio UI exited. Check /kaggle/working/studio-ui.log")
    try:
        urllib.request.urlopen("http://127.0.0.1:3791/", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError("Studio UI did not become ready. Check studio-ui.log")

# Expose the actual YuE2 Studio web UI; Vite proxies its API routes to the Rust service on 8791.
cloudflared = bundle / "cloudflared"
tunnel_log = open(WORK / "cloudflared.log", "w")
tunnel = subprocess.Popen([str(cloudflared), "tunnel", "--url", "http://127.0.0.1:3791", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
print("\nYuE2 Studio 5.0.0 is starting its Cloudflare Quick Tunnel.")
print("Keep this Kaggle cell running while you use the Studio.\n")
try:
    for line in tunnel.stdout:
        print(line.rstrip(), flush=True)
        tunnel_log.write(line)
        tunnel_log.flush()
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\nOPEN YOUR YUe2 STUDIO:", match.group(0), "\n", flush=True)
except KeyboardInterrupt:
    print("Stopping Studio processes...")
finally:
    for proc in (tunnel, ui, server):
        if proc and proc.poll() is None:
            proc.terminate()
    print("Studio stopped. Your library and generated files remain in:", DATA)
